# 02 - Build Silver Supply Chain Tables

        Build the Silver layer from the external-catalog source tables.

        The notebook reads only required columns, uses source keys for joins, pre-aggregates receipts and invoices by `PO_LINE_ID`, and writes managed tables into the AIDP standard catalog.

In [ ]:
# Edit these values for your AIDP workspace.
SOURCE_CATALOG = "aidp_sc_demo_source"
SOURCE_SCHEMA = "aidp_sc_demo"  # Use AIDP_SC_DEMO if your workspace exposes uppercase schema names.

TARGET_CATALOG = "aidp_sc_demo_standard"
SILVER_SCHEMA = "demo_supply_chain_silver"
GOLD_SCHEMA = "demo_supply_chain_gold"

# AIDP standard catalogs use managed Delta tables. Keep this as "delta" unless your tenancy requires a different table format.
TABLE_FORMAT = "delta"

In [ ]:
import re
from pyspark.sql import functions as F

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def qident(value: str) -> str:
    """Quote and validate a catalog, schema, or table identifier."""
    if not IDENTIFIER_PATTERN.fullmatch(value):
        raise ValueError(f"Unsupported identifier: {value!r}")
    return f"`{value}`"


def qname(*parts: str) -> str:
    return ".".join(qident(part) for part in parts)


def show_small(df, n: int = 20) -> None:
    """Display a small result in notebook UI, falling back to show()."""
    try:
        display(df.limit(n))
    except NameError:
        df.show(n, truncate=False)


SOURCE_TABLES = {
    "suppliers": "aidp_sc_suppliers",
    "supplier_sites": "aidp_sc_supplier_sites",
    "item_categories": "aidp_sc_item_categories",
    "items": "aidp_sc_items",
    "po_headers": "aidp_sc_po_headers",
    "po_lines": "aidp_sc_po_lines",
    "blanket_prices": "aidp_sc_blanket_prices",
    "receipts": "aidp_sc_receipts",
    "invoice_lines": "aidp_sc_invoice_lines",
}


def source_table(key: str):
    return spark.table(qname(SOURCE_CATALOG, SOURCE_SCHEMA, SOURCE_TABLES[key]))


def normalize_table_name(table: str) -> str:
    """AIDP standard catalog table names are easiest to resolve consistently in lowercase."""
    normalized = table.lower()
    if not IDENTIFIER_PATTERN.fullmatch(normalized):
        raise ValueError(f"Unsupported table identifier: {table!r}")
    return normalized


def target_table(schema: str, table: str) -> str:
    return qname(TARGET_CATALOG, schema, normalize_table_name(table))


def ensure_schema(schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qname(TARGET_CATALOG, schema)}")


def use_target_schema(schema: str) -> None:
    """Set the active catalog/schema before standard-catalog table access.

    AIDP notebooks can be stricter about three-part table resolution than local Spark.
    Setting the active catalog and schema before reads/writes avoids losing the target
    catalog during Delta table lookup.
    """
    spark.sql(f"USE CATALOG {qident(TARGET_CATALOG)}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(schema)}")
    spark.sql(f"USE SCHEMA {qident(schema)}")


def read_managed_table(schema: str, table: str):
    table_name = normalize_table_name(table)
    use_target_schema(schema)
    return spark.table(qident(table_name))


def write_managed_table(df, schema: str, table: str) -> None:
    """Overwrite one managed demo table in the AIDP standard catalog."""
    table_name = normalize_table_name(table)
    use_target_schema(schema)
    spark.sql(f"DROP TABLE IF EXISTS {qident(table_name)}")
    writer = df.write.mode("overwrite").option("overwriteSchema", "true")
    if TABLE_FORMAT:
        writer = writer.format(TABLE_FORMAT)
    writer.saveAsTable(qident(table_name))
    print(f"Wrote {target_table(schema, table_name)}")

## Load only required columns from the source tables

In [ ]:
suppliers = source_table("suppliers").select(
    F.col("SUPPLIER_ID").cast("long").alias("supplier_id"),
    F.col("SUPPLIER_NUMBER").alias("supplier_number"),
    F.col("SUPPLIER_NAME").alias("supplier_name"),
    F.col("STATUS").alias("supplier_status"),
    F.col("DEFAULT_CURRENCY").alias("supplier_default_currency"),
)

supplier_sites = source_table("supplier_sites").select(
    F.col("SUPPLIER_SITE_ID").cast("long").alias("supplier_site_id"),
    F.col("SUPPLIER_ID").cast("long").alias("supplier_id"),
    F.col("SITE_CODE").alias("supplier_site"),
    F.col("COUNTRY_CODE").alias("supplier_site_country"),
    F.col("CURRENCY_CODE").alias("supplier_site_currency"),
)

categories = source_table("item_categories").select(
    F.col("CATEGORY_ID").cast("long").alias("category_id"),
    F.col("CATEGORY_CODE").alias("category_code"),
    F.col("CATEGORY_NAME").alias("category_name"),
)

items = source_table("items").select(
    F.col("ITEM_ID").cast("long").alias("item_id"),
    F.col("ITEM_NUMBER").alias("item_number"),
    F.col("ITEM_DESCRIPTION").alias("item_description"),
    F.col("CATEGORY_ID").cast("long").alias("category_id"),
    F.col("PRIMARY_UOM").alias("primary_uom"),
    F.col("BASE_UNIT_PRICE").cast("double").alias("base_unit_price"),
)

po_headers = source_table("po_headers").select(
    F.col("PO_HEADER_ID").cast("long").alias("po_header_id"),
    F.col("PO_NUMBER").alias("po_number"),
    F.col("SUPPLIER_ID").cast("long").alias("supplier_id"),
    F.col("SUPPLIER_SITE_ID").cast("long").alias("supplier_site_id"),
    F.col("BUYER_NAME").alias("buyer_name"),
    F.col("CURRENCY_CODE").alias("currency_code"),
    F.col("ORDER_DATE").cast("date").alias("order_date"),
    F.col("PO_STATUS").alias("po_status"),
)

po_lines = source_table("po_lines").select(
    F.col("PO_LINE_ID").cast("long").alias("po_line_id"),
    F.col("PO_HEADER_ID").cast("long").alias("po_header_id"),
    F.col("LINE_NUMBER").cast("int").alias("line_number"),
    F.col("ITEM_ID").cast("long").alias("item_id"),
    F.col("ORDERED_QUANTITY").cast("double").alias("ordered_quantity"),
    F.col("UOM_CODE").alias("uom_code"),
    F.col("UNIT_PRICE").cast("double").alias("actual_unit_price"),
    F.col("NEED_BY_DATE").cast("date").alias("need_by_date"),
)

blanket_prices = source_table("blanket_prices").select(
    F.col("BLANKET_PRICE_ID").cast("long").alias("blanket_price_id"),
    F.col("SUPPLIER_ID").cast("long").alias("supplier_id"),
    F.col("SUPPLIER_SITE_ID").cast("long").alias("supplier_site_id"),
    F.col("ITEM_ID").cast("long").alias("item_id"),
    F.col("UOM_CODE").alias("uom_code"),
    F.col("CURRENCY_CODE").alias("currency_code"),
    F.col("UNIT_PRICE").cast("double").alias("blanket_unit_price"),
    F.col("EFFECTIVE_START_DATE").cast("date").alias("blanket_effective_start_date"),
    F.col("EFFECTIVE_END_DATE").cast("date").alias("blanket_effective_end_date"),
)

receipts = source_table("receipts").select(
    F.col("PO_LINE_ID").cast("long").alias("po_line_id"),
    F.col("RECEIPT_DATE").cast("date").alias("receipt_date"),
    F.col("RECEIVED_QUANTITY").cast("double").alias("received_quantity"),
    F.col("REJECTED_QUANTITY").cast("double").alias("rejected_quantity"),
    F.col("RETURNED_QUANTITY").cast("double").alias("returned_quantity"),
    F.col("RECEIPT_STATUS").alias("receipt_status_raw"),
)

invoice_lines = source_table("invoice_lines").select(
    F.col("PO_LINE_ID").cast("long").alias("po_line_id"),
    F.col("INVOICE_DATE").cast("date").alias("invoice_date"),
    F.col("INVOICED_QUANTITY").cast("double").alias("invoiced_quantity"),
    F.col("INVOICE_UNIT_PRICE").cast("double").alias("invoice_unit_price"),
    F.col("MATCH_STATUS").alias("invoice_match_status_raw"),
)

## Build key-based aggregations

In [ ]:
receipts_by_line = receipts.groupBy("po_line_id").agg(
    F.count(F.lit(1)).alias("receipt_count"),
    F.sum("received_quantity").alias("total_received_quantity"),
    F.sum("rejected_quantity").alias("total_rejected_quantity"),
    F.sum("returned_quantity").alias("total_returned_quantity"),
    F.max("receipt_date").alias("last_receipt_date"),
    F.max(F.when(F.col("rejected_quantity") > 0, F.lit(1)).otherwise(F.lit(0))).alias("has_rejected_receipt"),
)

invoices_by_line = invoice_lines.groupBy("po_line_id").agg(
    F.count(F.lit(1)).alias("invoice_line_count"),
    F.sum("invoiced_quantity").alias("total_invoiced_quantity"),
    F.min("invoice_unit_price").alias("invoice_unit_price_min"),
    F.max("invoice_unit_price").alias("invoice_unit_price_max"),
    F.max("invoice_date").alias("last_invoice_date"),
    F.max(F.when(F.col("invoice_match_status_raw") == "PRICE_MISMATCH", F.lit(1)).otherwise(F.lit(0))).alias("has_invoice_price_mismatch"),
    F.max(F.when(F.col("invoice_match_status_raw") == "QUANTITY_MISMATCH", F.lit(1)).otherwise(F.lit(0))).alias("has_invoice_quantity_mismatch"),
)

print("Prepared receipt and invoice aggregations by PO_LINE_ID.")

## Build the PO line comparison grain

In [ ]:
line_base = (
    po_lines.alias("pl")
    .join(po_headers.alias("ph"), F.col("pl.po_header_id") == F.col("ph.po_header_id"), "inner")
    .join(suppliers.alias("s"), F.col("ph.supplier_id") == F.col("s.supplier_id"), "inner")
    .join(supplier_sites.alias("ss"), F.col("ph.supplier_site_id") == F.col("ss.supplier_site_id"), "inner")
    .join(items.alias("i"), F.col("pl.item_id") == F.col("i.item_id"), "inner")
    .join(categories.alias("c"), F.col("i.category_id") == F.col("c.category_id"), "inner")
    .select(
        F.col("pl.po_line_id"),
        F.col("pl.po_header_id"),
        F.col("ph.po_number"),
        F.col("pl.line_number"),
        F.col("ph.order_date"),
        F.col("pl.need_by_date"),
        F.col("ph.po_status"),
        F.col("ph.buyer_name"),
        F.col("ph.supplier_id"),
        F.col("s.supplier_number"),
        F.col("s.supplier_name"),
        F.col("s.supplier_status"),
        F.col("s.supplier_default_currency"),
        F.col("ph.supplier_site_id"),
        F.col("ss.supplier_site"),
        F.col("ss.supplier_site_country"),
        F.col("ss.supplier_site_currency"),
        F.col("pl.item_id"),
        F.col("i.item_number"),
        F.col("i.item_description"),
        F.col("i.primary_uom"),
        F.col("i.base_unit_price"),
        F.col("i.category_id"),
        F.col("c.category_code"),
        F.col("c.category_name"),
        F.col("pl.ordered_quantity"),
        F.col("pl.uom_code"),
        F.col("ph.currency_code"),
        F.col("pl.actual_unit_price"),
    )
)

show_small(line_base.select("po_number", "line_number", "supplier_name", "item_number", "actual_unit_price"), 10)

## Resolve the applicable blanket price using equality keys plus effective dates

In [ ]:
from pyspark.sql import Window

blanket_candidates = (
    line_base.alias("lb")
    .join(
        blanket_prices.alias("bp"),
        (F.col("lb.supplier_id") == F.col("bp.supplier_id"))
        & (F.col("lb.supplier_site_id") == F.col("bp.supplier_site_id"))
        & (F.col("lb.item_id") == F.col("bp.item_id"))
        & (F.col("lb.uom_code") == F.col("bp.uom_code"))
        & (F.col("lb.currency_code") == F.col("bp.currency_code"))
        & (F.col("lb.order_date").between(F.col("bp.blanket_effective_start_date"), F.col("bp.blanket_effective_end_date"))),
        "left",
    )
    .select(
        F.col("lb.po_line_id"),
        F.col("bp.blanket_price_id"),
        F.col("bp.blanket_unit_price"),
        F.col("bp.blanket_effective_start_date"),
        F.col("bp.blanket_effective_end_date"),
    )
)

blanket_window = Window.partitionBy("po_line_id").orderBy(
    F.col("blanket_effective_start_date").desc_nulls_last(),
    F.col("blanket_price_id").asc_nulls_last(),
)

blanket_by_line = (
    blanket_candidates
    .withColumn("blanket_rank", F.row_number().over(blanket_window))
    .where(F.col("blanket_rank") == 1)
    .drop("blanket_rank")
)

## Write Silver comparison and feature tables

In [ ]:
comparison_pre_history = (
    line_base.alias("lb")
    .join(blanket_by_line.alias("bp"), "po_line_id", "left")
    .join(receipts_by_line.alias("r"), "po_line_id", "left")
    .join(invoices_by_line.alias("inv"), "po_line_id", "left")
    .withColumn(
        "receipt_signal",
        F.when(F.col("receipt_count").isNull(), F.lit("NOT_FOUND"))
        .when(F.col("total_rejected_quantity") > 0, F.lit("REJECTED"))
        .when(F.col("total_received_quantity") < F.col("ordered_quantity"), F.lit("PARTIAL"))
        .otherwise(F.lit("RECEIVED")),
    )
    .withColumn(
        "invoice_signal",
        F.when(F.col("invoice_line_count").isNull(), F.lit("NOT_FOUND"))
        .when((F.col("has_invoice_price_mismatch") == 1) | (F.abs(F.col("invoice_unit_price_max") - F.col("actual_unit_price")) > 0.01), F.lit("PRICE_MISMATCH"))
        .when(F.col("has_invoice_quantity_mismatch") == 1, F.lit("QUANTITY_MISMATCH"))
        .otherwise(F.lit("MATCHED")),
    )
)

history_window = (
    Window.partitionBy("supplier_id", "supplier_site_id", "item_id", "uom_code", "currency_code")
    .orderBy("order_date", "po_header_id", "po_line_id")
    .rowsBetween(-10, -1)
)

comparison = (
    comparison_pre_history
    .withColumn("supplier_item_prior_purchase_count", F.count("po_line_id").over(history_window))
    .withColumn("supplier_item_prior_avg_unit_price", F.round(F.avg("actual_unit_price").over(history_window), 2))
)

# Materialize once and reuse it for both Silver outputs.
comparison_materialized = comparison.cache()
comparison_materialized.count()

write_managed_table(comparison_materialized, SILVER_SCHEMA, "SUPPLY_PO_LINE_COMPARISON")

features = (
    comparison_materialized
    .withColumn("is_uom_comparable", F.col("uom_code") == F.col("primary_uom"))
    .withColumn("is_currency_comparable", F.col("currency_code") == F.col("supplier_site_currency"))
    .withColumn(
        "reference_unit_price",
        F.when((F.col("uom_code") == F.col("primary_uom")) & (F.col("currency_code") == F.col("supplier_site_currency")), F.coalesce("blanket_unit_price", "supplier_item_prior_avg_unit_price", "base_unit_price")),
    )
    .withColumn(
        "reference_source",
        F.when(F.col("blanket_unit_price").isNotNull(), F.lit("BLANKET_PRICE"))
        .when(F.col("supplier_item_prior_avg_unit_price").isNotNull(), F.lit("SUPPLIER_ITEM_HISTORY"))
        .when(F.col("base_unit_price").isNotNull(), F.lit("ITEM_BASE_PRICE"))
        .otherwise(F.lit("NONE")),
    )
    .withColumn("price_delta_amount", F.round(F.col("actual_unit_price") - F.col("reference_unit_price"), 2))
    .withColumn("price_delta_percent", F.round((F.col("price_delta_amount") / F.col("reference_unit_price")) * F.lit(100.0), 2))
    .withColumn(
        "price_signal",
        F.when(~F.col("is_uom_comparable") | ~F.col("is_currency_comparable"), F.lit("NOT_COMPARABLE"))
        .when(F.col("reference_unit_price").isNull(), F.lit("NO_REFERENCE"))
        .when(F.col("price_delta_percent") >= 15.0, F.lit("HIGH"))
        .when(F.col("price_delta_percent") <= -15.0, F.lit("LOW"))
        .otherwise(F.lit("NORMAL")),
    )
)

write_managed_table(features, SILVER_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_FEATURES")

## Confirm Silver outputs

In [ ]:
for table_name in ["SUPPLY_PO_LINE_COMPARISON", "SUPPLY_PO_PRICE_REVIEW_FEATURES"]:
    full_name = target_table(SILVER_SCHEMA, table_name)
    count_value = read_managed_table(SILVER_SCHEMA, table_name).count()
    print(f"{full_name}: {count_value:,} rows")

show_small(read_managed_table(SILVER_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_FEATURES").select(
    "po_number", "line_number", "actual_unit_price", "reference_unit_price", "price_signal", "receipt_signal", "invoice_signal"
).where(F.col("po_number").startswith("PO-DEMO-")), 20)